
OpenFDA Drug/Label – Signal Validation Script
==============================================
Scopo: verificare se un adverse event (AE) rilevato da un algoritmo di
farmacovigilanza (PRR, ROR, BCPNN, MGPS …) è già menzionato nel bugiardino
ufficiale FDA del farmaco, così da consolidare (o contestualizzare) la
significatività del segnale.

Logica di validation:
  - KNOWN          → l'AE compare nel bugiardino (segnale già riconosciuto)
  - POTENTIALLY_NEW → l'AE NON compare nel bugiardino (segnale candidato a nuovo)
  - NO_LABEL        → label non trovato per quel farmaco

Sezioni:
  1. Chiamata API base + recupero label
  2. Estrazione sezioni rilevanti dal bugiardino
  3. Ricerca testuale di un AE nelle sezioni
  4. Batch validation su una lista di segnali (drug, ae)
  5. Esplorazione / ispezione interattiva del label
  6. Esempi URL diretti all'API



In [ ]:

import requests
import json
import time
from typing import Optional


In [ ]:
#costanti 
BASE_URL = "https://api.fda.gov/drug/label.json"

# Sezioni del bugiardino più rilevanti per gli adverse events
SAFETY_SECTIONS = [
    "adverse_reactions",
    "warnings",
    "warnings_and_cautions",
    "boxed_warning",
    "precautions",
    "contraindications",
    "drug_interactions",
    "use_in_specific_populations",
]



In [ ]:

# 1. CHIAMATA API BASE


def fetch_label(drug_name: str, limit: int = 1, api_key: Optional[str] = None) -> Optional[dict]:
    """
    Recupera il/i bugiardino/i di un farmaco tramite openFDA drug/label API.

    Parametri
    ---------
    drug_name : str
        Nome generico o commerciale del farmaco (es. "lapatinib", "ibuprofen").
    limit : int
        Numero massimo di risultati (default 1 = label più recente).
    api_key : str, optional
        Chiave API openFDA (opzionale; senza chiave il rate limit è 240 req/min).
        Richiedila gratis su https://open.fda.gov/apis/authentication/

    Ritorna
    -------
    dict con i risultati openFDA, oppure None se la richiesta fallisce.

    Note sull'API
    -------------
    - URL base : https://api.fda.gov/drug/label.json
    - Parametri principali:
        search  → query in formato Lucene (es. openfda.generic_name:lapatinib)
        limit   → quanti risultati restituire (max 1000)
        skip    → offset per paginazione
        count   → campo su cui contare (invece di restituire i record)
    - Campi searchable: adverse_reactions, warnings, boxed_warning,
      contraindications, precautions, openfda.generic_name, openfda.brand_name …
    - Documentazione: https://open.fda.gov/apis/drug/label/
    """
    params = {
        # Cerca nel nome generico (openfda.generic_name) O nel nome commerciale
        "search": (
            f'openfda.generic_name:"{drug_name}" '
            f'OR openfda.brand_name:"{drug_name}"'
        ),
        "limit": limit,
    }
    if api_key:
        params["api_key"] = api_key

    try:
        resp = requests.get(BASE_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        total = data["meta"]["results"]["total"]
        returned = len(data["results"])
        print(f"[OK] '{drug_name}' → {total} label totali trovati, restituiti: {returned}.")
        return data

    except requests.exceptions.HTTPError as e:
        status = resp.status_code if resp is not None else "?"
        if status == 404:
            print(f"[WARN] Nessun label trovato per '{drug_name}'.")
        else:
            print(f"[ERR] HTTP {status}: {e}")
        return None
    except requests.exceptions.ConnectionError:
        print(f"[ERR] Impossibile connettersi a {BASE_URL}. Verifica la connessione.")
        return None
    except Exception as e:
        print(f"[ERR] Errore generico: {e}")
        return None

# 2. ESTRAZIONE SEZIONI RILEVANTI


def extract_safety_sections(label_record: dict) -> dict:
    """
    Estrae le sezioni di sicurezza da un singolo record label openFDA.

    Ogni campo restituito dall'API è una lista di stringhe; qui viene
    concatenato e portato in minuscolo per la ricerca testuale.

    Ritorna un dizionario { nome_sezione: testo_concatenato_lowercase }.
    """
    sections = {}
    for field in SAFETY_SECTIONS:
        content = label_record.get(field)
        if content:
            sections[field] = " ".join(content).lower()
    return sections


def print_label_info(label_record: dict) -> None:
    """Stampa le informazioni identificative del label."""
    openfda = label_record.get("openfda", {})
    print("\n  ┌─ LABEL INFO ─────────────────────────────────────────┐")
    print(f"  │  Brand name   : {openfda.get('brand_name', ['N/A'])[:1]}")
    print(f"  │  Generic name : {openfda.get('generic_name', ['N/A'])[:1]}")
    print(f"  │  Manufacturer : {openfda.get('manufacturer_name', ['N/A'])[:1]}")
    print(f"  │  Route        : {openfda.get('route', ['N/A'])[:1]}")
    print(f"  │  Effective    : {label_record.get('effective_time', 'N/A')}")
    print("  └──────────────────────────────────────────────────────┘")


def list_all_sections(label_record: dict) -> None:
    """Stampa tutte le sezioni presenti nel record (utile per l'esplorazione)."""
    print("\n  Sezioni presenti nel label:")
    for key, val in label_record.items():
        if key == "openfda":
            continue
        if isinstance(val, list) and val:
            preview = str(val[0])[:100].replace("\n", " ")
            print(f"    [{key}]  {preview}…")

# 3. RICERCA DI UN AE NELLE SEZIONI

def search_ae_in_label(ae_term: str, sections: dict, verbose: bool = True) -> dict:
    """
    Cerca un adverse event nelle sezioni di sicurezza estratte dal bugiardino.

    Esegue una ricerca testuale semplice (substring match).
    Per una ricerca più robusta con sinonimi/MedDRA hierarchy si può
    ampliare il termine con varianti (es. "diarrhoea" + "diarrhea").

    Parametri
    ---------
    ae_term  : str  - termine da cercare (es. "diarrhea", "hepatotoxicity")
    sections : dict - output di extract_safety_sections()
    verbose  : bool - stampa i risultati a video

    Ritorna
    -------
    dict con:
      found        : bool   - trovato in almeno una sezione
      sections_hit : list   - sezioni dove compare
      snippets     : dict   - { sezione: estratto contestuale ±150 char }
    """
    ae_lower = ae_term.lower().strip()
    result = {"found": False, "sections_hit": [], "snippets": {}}

    for section_name, text in sections.items():
        if ae_lower in text:
            result["found"] = True
            result["sections_hit"].append(section_name)
            # Estratto contestuale (±150 caratteri attorno al match)
            idx   = text.find(ae_lower)
            start = max(0, idx - 150)
            end   = min(len(text), idx + len(ae_lower) + 150)
            snippet = "…" + text[start:end].replace("\n", " ") + "…"
            result["snippets"][section_name] = snippet

    if verbose:
        status = "✅ TROVATO" if result["found"] else "❌ NON TROVATO"
        print(f"\n    AE: '{ae_term}'  →  {status}")
        for sec in result["sections_hit"]:
            print(f"      Sezione : [{sec}]")
            print(f"      Snippet : {result['snippets'][sec][:280]}")

    return result



# 4. BATCH VALIDATION SU LISTA DI SEGNALI

def validate_signals(signals: list[dict], api_key: Optional[str] = None) -> list[dict]:
    """
    Valida una lista di segnali rispetto ai bugiardini openFDA.

    Input
    -----
    signals  : lista di dict con campi:
                 drug (str) - nome del farmaco
                 ae   (str) - adverse event (MedDRA preferred term o testo libero)
    api_key  : str, optional - chiave API openFDA

    Output
    ------
    Lista di dict arricchiti con:
      label_found       : bool - label recuperato con successo
      ae_in_label       : bool - AE trovato nel bugiardino
      sections_hit      : list - sezioni dove compare
      validation_status : str  - "KNOWN" | "POTENTIALLY_NEW" | "NO_LABEL"

    Interpretazione
    ---------------
      KNOWN          → l'AE è già riconosciuto nel label ufficiale:
                       il segnale algoritmico è confermato dalla label evidence.
      POTENTIALLY_NEW → l'AE non figura nel label: potenziale segnale nuovo,
                       merita approfondimento (case-by-case, letteratura …).
      NO_LABEL        → nessun label trovato per quel farmaco su openFDA.
    """
    results      = []
    label_cache  = {}  # evita chiamate duplicate per lo stesso farmaco

    for sig in signals:
        drug = sig["drug"]
        ae   = sig["ae"]
        print(f"\n  {'─'*60}")
        print(f"  Drug: {drug}  │  AE: {ae}")

        # Recupero label (con cache per-farmaco)
        if drug not in label_cache:
            data = fetch_label(drug, limit=1, api_key=api_key)
            label_cache[drug] = data
        else:
            data = label_cache[drug]
            print(f"  (label da cache per '{drug}')")

        row = {
            **sig,
            "label_found"      : False,
            "ae_in_label"      : False,
            "sections_hit"     : [],
            "validation_status": "NO_LABEL",
        }

        if data and data.get("results"):
            row["label_found"] = True
            record   = data["results"][0]
            sections = extract_safety_sections(record)

            ae_result            = search_ae_in_label(ae, sections, verbose=True)
            row["ae_in_label"]   = ae_result["found"]
            row["sections_hit"]  = ae_result["sections_hit"]
            row["validation_status"] = (
                "KNOWN" if ae_result["found"] else "POTENTIALLY_NEW"
            )
        else:
            print("  [SKIP] Nessun label disponibile.")

        results.append(row)
        time.sleep(0.25)  # rispetta rate limit openFDA (240 req/min senza API key)

    return results


def print_validation_summary(results: list[dict]) -> None:
    """Stampa una tabella riassuntiva dei risultati di validation."""
    print(f"\n{'═'*74}")
    print(f"  RIEPILOGO VALIDATION  ({len(results)} segnali)")
    print(f"{'═'*74}")
    print(f"  {'Drug':<20} {'AE':<33} {'Status':<18} Sezioni")
    print(f"  {'─'*20} {'─'*33} {'─'*18} {'─'*20}")
    for r in results:
        secs = ", ".join(r["sections_hit"]) if r["sections_hit"] else "—"
        print(f"  {r['drug']:<20} {r['ae']:<33} {r['validation_status']:<18} {secs}")

    known = sum(1 for r in results if r["validation_status"] == "KNOWN")
    new   = sum(1 for r in results if r["validation_status"] == "POTENTIALLY_NEW")
    nolab = sum(1 for r in results if r["validation_status"] == "NO_LABEL")
    print(f"\n  ✅ KNOWN: {known}  │  🔍 POTENTIALLY NEW: {new}  │  ⚠️  NO LABEL: {nolab}")
    print(f"{'═'*74}\n")


# 5. ESPLORAZIONE INTERATTIVA


def explore_label(drug_name: str, api_key: Optional[str] = None) -> None:
    """
    Modalità esplorativa: mostra tutte le sezioni disponibili
    e il testo delle sezioni di sicurezza di un farmaco.
    Utile per capire cosa contiene il label prima di validare.
    """
    data = fetch_label(drug_name, limit=1, api_key=api_key)
    if not data or not data.get("results"):
        print("Nessun label da esplorare.")
        return

    record = data["results"][0]
    print_label_info(record)
    list_all_sections(record)

    print("\n  ── Testo sezioni di sicurezza ──")
    sections = extract_safety_sections(record)
    for name, text in sections.items():
        print(f"\n  [{name.upper()}]")
        print("  " + text[:700].replace("\n", "\n  ") + ("…" if len(text) > 700 else ""))


# 6. ESEMPI URL DIRETTI ALL'API


def print_api_examples() -> None:
    """
    Mostra URL di esempio per esplorare l'API direttamente dal browser
    o da curl/Postman — utile per il debug e la prototipazione.
    """
    base = BASE_URL
    examples = [
        ("Cerca per nome generico",
         f"{base}?search=openfda.generic_name:lapatinib&limit=1"),

        ("Cerca per nome commerciale (Tykerb = lapatinib)",
         f"{base}?search=openfda.brand_name:tykerb&limit=1"),

        ("Label che menzioni 'diarrhea' nelle adverse reactions",
         f"{base}?search=openfda.generic_name:lapatinib+AND+adverse_reactions:diarrhea&limit=1"),

        ("Conta quali farmaci menzionano 'hepatotoxicity'",
         f"{base}?search=adverse_reactions:hepatotoxicity&count=openfda.generic_name.exact"),

        ("Storico label lapatinib (ultime 5 versioni)",
         f"{base}?search=openfda.generic_name:lapatinib&limit=5"),

        ("Cerca in boxed_warning (black box)",
         f"{base}?search=openfda.generic_name:lapatinib+AND+boxed_warning:hepatotoxicity&limit=1"),

        ("Cerca per NDC (National Drug Code)",
         f"{base}?search=openfda.package_ndc:0007-4692*&limit=1"),

        ("Cerca ibuprofen con GI bleeding nelle warnings",
         f"{base}?search=openfda.generic_name:ibuprofen+AND+warnings:\"gastrointestinal+bleeding\"&limit=1"),
    ]

    print("\n  ── Esempi di URL API openFDA drug/label ──")
    print("  (copia-incolla nel browser o in curl per testare)\n")
    for desc, url in examples:
        print(f"  # {desc}")
        print(f"  {url}\n")




In [ ]:
# MAIN – DEMO


if __name__ == "__main__":

    # ── Configurazione ────────────────────────────────────────────────────────
    # Inserisci qui la tua API key openFDA (opzionale, aumenta il rate limit)
    # Richiedila gratis su https://open.fda.gov/apis/authentication/
    API_KEY = None  # es. "AbCdEfGhIjKlMnOpQrSt1234"

    print("=" * 74)
    print("  OPENFDA DRUG/LABEL – SIGNAL VALIDATION")
    print("=" * 74)

    # ── Step 1: Esempi di URL API ─────────────────────────────────────────────
    print("\n[STEP 1] Esempi di URL per chiamate dirette all'API")
    print_api_examples()

    # ── Step 2: Esplorazione label ────────────────────────────────────────────
    print("\n[STEP 2] Esplorazione label lapatinib")
    explore_label("lapatinib", api_key=API_KEY)

    # ── Step 3: Batch validation ──────────────────────────────────────────────
    # ↓ PERSONALIZZA QUESTA LISTA con i tuoi segnali algoritmici (PRR/ROR/ecc.)
    print("\n[STEP 3] Batch validation segnali")

    my_signals = [
        # Segnali attesi KNOWN (presenti nel label FDA di lapatinib)
        {"drug": "lapatinib", "ae": "diarrhea"},
        {"drug": "lapatinib", "ae": "rash"},
        {"drug": "lapatinib", "ae": "hepatotoxicity"},
        {"drug": "lapatinib", "ae": "nausea"},
        # Segnali potenzialmente nuovi (non nel label standard – da Cerbito et al.)
        {"drug": "lapatinib", "ae": "hypocapnia"},
        {"drug": "lapatinib", "ae": "lip ulceration"},
        {"drug": "lapatinib", "ae": "hepatic infection"},
        # Secondo farmaco – demo
        {"drug": "ibuprofen", "ae": "gastrointestinal bleeding"},
        {"drug": "ibuprofen", "ae": "hallucination"},
    ]

    results = validate_signals(my_signals, api_key=API_KEY)

    # ── Step 4: Riepilogo ─────────────────────────────────────────────────────
    print_validation_summary(results)

    # ── Step 5: Salva in JSON ─────────────────────────────────────────────────
    output_path = "signal_validation_results.json"
    with open(output_path, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Risultati salvati in: {output_path}")